In [ ]:
from sympy import symbols, Matrix, Function, Symbol, pprint, diff, Eq, MatMul, Derivative, Determinant, MatrixSymbol, solve, Identity, eye, exp, integrate, diag, lambdify
from sympy.physics.mechanics import dynamicsymbols, ReferenceFrame, Point, Particle, KanesMethod, init_vprinting
from IPython.display import display
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
init_vprinting()

# Higher Order ODE Systems modeled with Linear Algebra

## System of N Coupled 1st order ODEs
each ODE has only 1st order dependency on time

In [ ]:
y, x= dynamicsymbols('y x')
t = symbols('t')

In [ ]:
x.diff()

In [ ]:
# Enable dot notation printing
init_vprinting()

# Define a time-dependent variable x(t)
x = dynamicsymbols('x')
t = Symbol('t')

# First derivative: prints as ẋ
display(x.diff(t))

# Second derivative: prints as ẍ
display(x.diff(t, 2))

In [ ]:
# system
n = Symbol('n')
x_0 = x
x_1 = x.diff(t)
x_2 = x.diff(t, 2)
x_3 = x.diff(t, 3)
# x_n = diff(x, (t, n))
x_vec = Matrix([[x_0, x_1, x_2, x_3]]).T
a_0, a_1, a_2, a_3 = symbols('a_0 a_1 a_2 a_3')
const_vec = Matrix([[a_0, a_1, a_2, a_3]]).T
# we can divide by a_3 so that the first constant is 1, this just makes our life easier
const_vec_subs = const_vec.subs({a_3: 1})
char_poly = (x_vec.T * const_vec_subs)[0]
char_eq = Eq(0, char_poly)
display("This is our characteristic polynomial")
display(char_eq)
# A = Matrix([[0, 1, 0], [0, 0, 1]

In [ ]:
A = Matrix([[0, 1, 0], [0, 0, 1], -const_vec.T[:,:-1]])
display(A)

In [ ]:
jacobian = Eq(x_vec[1:,:],MatMul(A,x_vec[0:-1,:]))

In [ ]:
jacobian.doit()

In [ ]:
# we can see from before that this expression for x_3 is correct
Eq(x_3, solve(char_eq, x_3)[0])

In [ ]:
# eigs of A are lambdas that satisfy
_lambda = Symbol('lambda')
A_sym = MatrixSymbol('A', *A.shape)
I = Identity(A.shape[0])
detr = Determinant(A_sym - _lambda * I)
detr_eq = Eq(0, detr)
display(detr_eq)

In [ ]:
A - _lambda * I

In [ ]:
# 3rd order ode is
# you can do this but it's cooler to use the same symbol I think
# A - _lambda * eye(A.shape[0])
Determinant(A - _lambda * I.as_explicit())

In [ ]:
A_poly = detr_eq.subs({A_sym: A, I:I.as_explicit()}).doit()
A_poly

In [ ]:
Eq(A_poly.rhs, char_eq.rhs)

In [ ]:
solve(Eq(A_poly.rhs, char_eq.rhs), _lambda)[0].subs({a_0:1, a_1:1, a_2:1, x: exp(t)})

In [ ]:
n = Symbol('n')
m = Symbol('m')
T_sym = MatrixSymbol('T', n,n) # invertible matrix tranformation that changes A
A_sym = MatrixSymbol('A', n,m)
v = MatrixSymbol("vbold", n, 1)
z = MatrixSymbol("z", n, 1)
D = MatrixSymbol("D", n, n) # diagonal matrix
trans_eq = Eq(A_sym*T_sym, T_sym * D)
display(trans_eq)

In [ ]:
Eq(T_sym * v, _lambda * v)

# Types of dynamical systems

# linear
Given vectors $x$ and $y$,
If $f(x)$ and $f(y)$ are solutions, then there is a solution $f(x)$ such that $f(z) = \alpha f(x) + \beta f(y)$ (where $\alpha$ and $\beta$ are scalars).

Linear dynamical systems can be described by this equation:
$$
\frac{d}{dt} x(t) = A x(t)
$$

In [ ]:
n=3
A_mat = MatrixSymbol("A", n, n)
x_v = Matrix([[Function('theta_0')(t), Function('theta_1')(t), Function('theta_2')(t)]]).T

In [ ]:
Eq(x_v.diff(), A_mat*x_v)

## weakly nonlinear
Fixed points, saddle points, etc

## chaotic
still deterministic but "sensitive dependence on initial conditions"

# Linearization
You focus in on fixed points (could be saddle points or not)

Let $\dot x = f(x)$ near $\bar x$ where $f(\bar x) = 0$. $\bar x$ is therefore a fixed point (a point where $\dot x = 0$).

We can examine the behavior of $f(x)$ around this point by looking at the perturbation due to a small change in x ($\Delta x$).

We are therefore analyzing a value $x_1$ where $x_1 = \bar x + \Delta x$, and we want to evaluate the change in this point $x_1$ over time.
$$
\frac{d}{dt} x_1 = \frac{d}{dt} \bar x + \frac{d}{dt} \Delta x
$$
$$
\frac{d}{dt} x_1 = 0 + \frac{d}{dt} \Delta x
$$
We only care about the expression on the right, so:
$$
\frac{d}{dt} \Delta x = f(\bar x + \Delta x)
$$

You might be tempted to say
$$
f(\bar x + \Delta x) = f(\bar x) + f(\Delta x)
$$
but
1. we don't actually know this and
2. the whole point is that we *don't* know what f(x) actually is for some arbitrary point - we only know it for $f(\bar x)$

What we can do is Taylor expand! Normally we just keep the first one, which is the Jacobian.
$$
\frac{d}{dt} \Delta x = f(\bar x) + \nabla f(\bar x) \Delta x + \frac{1}{2}\nabla ^2 f(\bar x) \Delta x ^2 + . . .
$$

$\Delta x$ HAS TO BE SMALL

The Jacobian is a matrix of partial differential equations which are much easier to calculate.
So for values near the fixed points, we get a simple expression that
$\dot x = A x$, where $A = \nabla f(\bar x)$

## why linearize?
we know lots about these kinds of systems

If we integrate this to solve for $x(t_0 + t)$, we get
$$
\frac{d}{dt}x = A * x
$$
$$
x(t_0 + t) = e^{At}x(t_0)
$$

We can remember that $t$ is the time since $t_0$ and simply write
$$
x(t) = e^{At}x(t_0)
$$

Now let's find a vector space $z$ that can be transformed from $x$ with a matrix $T$ such that it has a diagonal Jacobian (i.e. changes in each direction are independent from each other)
$$
x = Tz
$$
$$
A = T \Lambda T^{-1}
$$
$$
AT = T \Lambda
$$
$$
\dot z = \Lambda z
$$

Now a funky thing can happen. We need to find a transformed expression for $e^{At}$. I don't want to write it all out but basically you
1. get the taylor expansion of $e^{At}$
2. plug in $T \Lambda T^{-1}$ for $A$
3. all the $T$ and $T^{-1}$ terms can be factored out to have an expression that looks like $T [\text{sum}] T^{-1}$
4. And that sum turns out to be exactly the Taylor expansion for $e^{\Lambda t}$

So we can rewrite this:
$$
x(t) = e^{At}x(t_0)
$$
as
$$
x(t) = T e^{\Lambda t} T^{-1} x(t_0)
$$

Why do we do this?

Because unlike $A$, $\Lambda$ is a diagonal matrix, making it much easier to use for computations.

In [ ]:
L_sym = diag(*symbols("lambda_1 lambda_2 lambda_3"))

In [ ]:
L_sym **2

In [ ]:
exp(L_sym)

In [ ]:
a_to_lambda = Eq(A, T * L * T.inv())

In [ ]:
a_to_lambda

In [ ]:
x = Function("x")(t)
dxdt = A_mat * x

In [ ]:
integrate(dxdt, (t, "t_0", t))

In [ ]:
x_bar = Symbol("xbar")
x_dot = Symbol("xdot")
x_del = Symbol(r"\Delta x")

In [ ]:
T = MatrixSymbol("T", 3,3,)
A = MatrixSymbol("A", 3,3,)
L = MatrixSymbol("Lambda", 3,3)

In [ ]:
Eq(A, T * L * T.inv())

In [ ]:
Eq(A*T, T * L)

In [ ]:
Eq(T.inv() * A*T, L)

In [ ]:
T.inv() * T

In [ ]:
T * T.inv()

In [ ]:
A * T * T.inv()

In [ ]:
T.inv() * A * T

In [ ]:
expr = ((T * L * T.inverse()) * (T * L * T.inverse())).simplify()

In [ ]:
T * (L**2) * 

In [ ]:
Eq(expr, T * L**2 * T.inverse())

In [ ]:
x = x_bar + x_del

# Simple(ish) Example of Nonlinearity - Duffing Equation
(2 well potential)

In [ ]:
x = Symbol("x")
v_x = (x**4 - 2*x**2)/4

In [ ]:
x_vals = np.linspace(-1.5,1.5,20)
well_potential = lambdify(x, v_x)

In [ ]:
well_potential(x_vals)

In [ ]:
sns.set_style("white")
sns.set_palette("viridis")
sns.lineplot(x=x_vals, y=well_potential(x_vals))

In [ ]:
t = Symbol("t")
x = Function("x")(t)
v = Symbol("v")
v_x = (x**4 - 2*x**2)/4
x_dot = v
v_dot = - v_x.diff(t)

In [ ]:
v_dot

In [ ]:
u_x = (x**4 - 2*x**2)/4

In [ ]:
v_dot = u_x.diff()

In [ ]:
v_dot.diff().subs({x:0})

In [ ]:
from sympy import jacobi

In [ ]:
F = Matrix([[v, x - x**3]]).T

In [ ]:
F.jacobian([x, v])

In [ ]:
m = 1
x = Symbol("x")
v = Function("v")(x)
KE  = 1/2 * m * v**2
U = (x**4 - 2*x**2)/4

In [ ]:
H = KE + U
H

In [ ]:
F = Matrix([[v], [- U.diff(x)]])

In [ ]:
J = - F.jacobian([x, v])

In [ ]:
fixed_point = Matrix([[0,0]])

In [ ]:
J

In [ ]:
first_point = J.subs({x:0, v:0})

In [ ]:
first_point.simplify()

In [ ]:
vecs = [i[-1] for i in first_point.eigenvects()]
vecs

In [ ]:
second_point = J.subs({x:1, v:0})
second_point.simplify()
second_point.eigenvects()

In [ ]:
second_point

In [ ]:
J.subs({x:1, v:0})

In [ ]:
J

In [ ]:
phase_matrix = Matrix([[x,v]]).T
x_0 = 0
v_0 = 1
xv_0 = Matrix([[x_0, v_0]])
first_point = J.subs({x:x_0, v:v_0})
x_t = exp(first_point*t)*xv_0.T

In [ ]:
x_t.simplify()

In [ ]:
from sympy import simplify

In [ ]:
times = np.linspace(0,10,11)
xv = [simplify(x_t.subs({t:i})) for i in times]

In [ ]:
xv

# How to draw?
Nonlinear ODE

In [ ]:
x = Symbol("x")
v = Symbol("v")
U = x**2/2 - x**3/3
ddx_ddt = - U.diff(x)
x_dot = v
v_dot = ddx_ddt
fixed_points = [Matrix([[0,0]]).T, Matrix([[1,0]]).T]

In [ ]:
x_vals = np.linspace(-1,2,100)
u_func = lambdify(x, U)

In [ ]:
plt.scatter(x=x_vals, y=u_func(x_vals))

In [ ]:
f_vec = Matrix([[v, v_dot]])

In [ ]:
J = f_vec.jacobian([x,v])

In [ ]:
j_subs * np.array([0,1]).reshape(2,1)

In [ ]:
for fixed_point in fixed_points:
    j_subs = J.subs({x:fixed_point[0], y:fixed_point[0]})
    display(j_subs)
    for _lambda, _, vec in j_subs.eigenvects():
        print(f"eigenvalue")
        display(_lambda)
        print(f"eigenvector")
        display(vec)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
x_min, x_max = (2,2)
v_min, v_max = (2,2)
Y, X = np.mgrid[-v_min:v_max:200j, -x_min:x_max:200j]

In [ ]:
X.flatten()[0]

In [ ]:
x_vels = []
y_vels = []
for x_i,y_i in zip(X.flatten(),Y.flatten()):
    phase_point = J.subs({x:x_i, y:y_i}) * np.array([[x_i,y_i]]).T
    x_vels.append(float(phase_point[0]))
    y_vels.append(float(phase_point[1]))

x_vels = np.array(x_vels).reshape(X.shape)
y_vels = np.array(y_vels).reshape(Y.shape)

In [ ]:
fixed_points = np.array([[0,0], [0,1], [0.5,0.5],[1,0.5], [1,1], [0.25,0],[0.5,0],[-1,0], [1,0]])

In [ ]:
fixed_points

In [ ]:
fig, axs = plt.subplots(figsize=(7, 9))

#  Varying density along a streamline
axs.streamplot(X, Y, x_vels, y_vels, 
               # start_points=fixed_points
              )

# Quadratic Potential

In [ ]:
x = Symbol("x")
v = Symbol("v")
U = - x**2/2 + x**4/4
ddx_ddt = - U.diff(x)
x_dot = v
v_dot = ddx_ddt
fixed_points = [Matrix([[0,0]]).T, Matrix([[1,0]]).T, Matrix([[-1,0]]).T]

In [ ]:
x_vals = np.linspace(-1.5,1.5,100)
u_func = lambdify(x, U)

In [ ]:
plt.scatter(x=x_vals, y=u_func(x_vals))

In [ ]:
f_vec = Matrix([[v, v_dot]])

In [ ]:
J = f_vec.jacobian([x,v])

In [ ]:
j_subs * np.array([0,1]).reshape(2,1)

In [ ]:
for fixed_point in fixed_points:
    j_subs = J.subs({x:fixed_point[0], y:fixed_point[0]})
    display(j_subs)
    for _lambda, _, vec in j_subs.eigenvects():
        print(f"eigenvalue")
        display(_lambda)
        print(f"eigenvector")
        display(vec)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
x_min, x_max = (-1.5,1.5)
v_min, v_max = (-1.5,1.5)
Y, X = np.mgrid[v_min:v_max:200j, x_min:x_max:200j]

In [ ]:
X.flatten()[0]

In [ ]:
x_vels = []
y_vels = []
for x_i,y_i in zip(X.flatten(),Y.flatten()):
    phase_point = J.subs({x:x_i, y:y_i}) * np.array([[x_i,y_i]]).T
    x_vels.append(float(phase_point[0]))
    y_vels.append(float(phase_point[1]))

x_vels = np.array(x_vels).reshape(X.shape)
y_vels = np.array(y_vels).reshape(Y.shape)

In [ ]:
fixed_points = np.array([[0,0], [0,1], [0.5,0.5],[1,0.5], [1,1], [0.25,0],[0.5,0],[-1,0], [1,0]])

In [ ]:
fixed_points

In [ ]:
fig, axs = plt.subplots(figsize=(6,6))

#  Varying density along a streamline
axs.streamplot(X, Y, x_vels, y_vels, 
               # start_points=fixed_points
              )

# Quadratic Potential with Friction

In [ ]:
x.diff(t)

In [ ]:
x = Symbol("x")
v = Symbol("v")
U = - x**2/2 + x**4/4
ddx_ddt = - U.diff(x) - v
x_dot = v
v_dot = ddx_ddt
fixed_points = [Matrix([[0,0]]).T, Matrix([[1,0]]).T, Matrix([[-1,0]]).T]

In [ ]:
x_vals = np.linspace(-1.5,1.5,100)
u_func = lambdify(x, U)

In [ ]:
plt.scatter(x=x_vals, y=u_func(x_vals))

In [ ]:
f_vec = Matrix([[v, v_dot]])

In [ ]:
J = f_vec.jacobian([x,v])

In [ ]:
j_subs * np.array([0,1]).reshape(2,1)

In [ ]:
for fixed_point in fixed_points:
    j_subs = J.subs({x:fixed_point[0], y:fixed_point[0]})
    display(j_subs)
    for _lambda, _, vec in j_subs.eigenvects():
        print(f"eigenvalue")
        display(_lambda)
        print(f"eigenvector")
        display(vec)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
x_min, x_max = (-1.5,1.5)
v_min, v_max = (-1.5,1.5)
Y, X = np.mgrid[v_min:v_max:200j, x_min:x_max:200j]

In [ ]:
X.flatten()[0]

In [ ]:
x_vels = []
y_vels = []
for x_i,y_i in zip(X.flatten(),Y.flatten()):
    phase_point = J.subs({x:x_i, y:y_i}) * np.array([[x_i,y_i]]).T
    x_vels.append(float(phase_point[0]))
    y_vels.append(float(phase_point[1]))

x_vels = np.array(x_vels).reshape(X.shape)
y_vels = np.array(y_vels).reshape(Y.shape)

In [ ]:
fixed_points = np.array([[0,0], [0,1], [0.5,0.5],[1,0.5], [1,1], [0.25,0],[0.5,0],[-1,0], [1,0]])

In [ ]:
fixed_points

In [ ]:
fig, axs = plt.subplots(figsize=(4,4))

#  Varying density along a streamline
axs.streamplot(X, Y, x_vels, y_vels, 
               # start_points=fixed_points
              )

# Hartman-Grobman Theorem


if all eigenvalues of A (evaluated at a fixed point) have nonzero real part (hyperbolic fixed points), then locally (for small neighborhood around x bar), then linearized dynamics (given by x dot = A x) are "topologically equivalent" to the nonlinear system.